# forex_rl_v4 — Colab GPU training (multi-contributor)

Runs a slice of the full walk-forward sweep (12 folds x 3 seeds = 36
fold/seed combos, covering all 16 years of data on disk) as concurrent
`train.py` processes on ONE Colab GPU, writing to a Google Drive folder
SHARED across everyone contributing — so multiple people's sessions combine
into one pool of results instead of each person's work being stuck on
their own Drive.

**Nothing to type, nothing to download or upload.** Open this from the
[Contribute tab](https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/) link, accept the Google Drive access
prompt the next cell shows you, then `Runtime -> Change runtime type ->
GPU` and `Runtime -> Run all` — that's the whole setup. Your name is
generated for you (and remembered on your Drive for next time), the repo
is public so the code and historical price data pull straight from
GitHub, and your results push themselves straight back to GitHub in the
background as you train. No dashboard runs here, and none needed —
everyone's progress, including yours, shows up at the ONE shared dashboard:

**[https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/](https://martin-gamcptrxwpxkksceimkyx3.streamlit.app/)**

**If the session disconnects:** re-run the notebook from the top (`Runtime
-> Run all`). You get the SAME generated name and the SAME shard indices
back, and each process's `--resume` skips fold/seed combos already
finished (by ANYONE, not just you) and resumes an interrupted fold from its
last mid-fold checkpoint. You'll lose whatever hadn't been auto-pushed yet
(up to ~5 minutes of progress), same as any other interruption.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Config — nothing to edit, just run this cell ===
# MY_NAME: auto-generated once and saved to YOUR OWN Drive (not the shared
# folder) so it's stable across reruns/reconnects — the claims registry and
# the dashboard's contributor leaderboard both key off this, so a name that
# changed every run would look like a new person abandoning old shards each
# time instead of one person resuming. Delete the file below if you ever
# want a fresh name.
import os
import random

_NAME_FILE = '/content/drive/MyDrive/.forex_rl_v4_my_name.txt'
if os.path.exists(_NAME_FILE):
    with open(_NAME_FILE) as f:
        MY_NAME = f.read().strip()
else:
    _ADJECTIVES = ['swift', 'sharp', 'bold', 'lucky', 'steady', 'silent', 'quick', 'wild',
                   'calm', 'fierce', 'golden', 'iron', 'midnight', 'rogue', 'stealth', 'turbo',
                   'rapid', 'clever', 'brave', 'sly']
    _NOUNS = ['falcon', 'bull', 'bear', 'pip', 'candle', 'trader', 'wolf', 'hawk', 'tiger',
              'shark', 'viper', 'eagle', 'ninja', 'comet', 'rocket', 'panther', 'cobra',
              'phoenix', 'lynx', 'scalper']
    MY_NAME = f'{random.choice(_ADJECTIVES)}-{random.choice(_NOUNS)}-{random.randint(10, 99)}'
    with open(_NAME_FILE, 'w') as f:
        f.write(MY_NAME)

# N_SHARDS_WANTED: how many shard slots to claim this session — one Colab
# GPU comfortably runs 4 in parallel (the project's standard allocation).
N_SHARDS_WANTED = 4

# TOTAL_SHARDS: total shard count across EVERYONE combined — the
# denominator for train.py's round-robin split. Must match what everyone
# else in the group is using. 36 = 12 folds x 3 seeds, i.e. one shard per
# fold/seed combo (the most parallelism this sweep can actually use).
TOTAL_SHARDS = 36

# SHARED_FOLDER: the group's shared Drive folder. Fixed for everyone — you
# get it by opening the link from the Contribute tab and clicking "Add
# shortcut to Drive", which makes it appear at this exact path in your own
# MyDrive with no manual configuration.
SHARED_FOLDER = '/content/drive/MyDrive/forex_rl_v4_shared_results'

# ITERATIONS: PPO iterations per fold/seed combo.
ITERATIONS = 100

print(f'You are: {MY_NAME}')
print(f'Wants {N_SHARDS_WANTED} of {TOTAL_SHARDS} total shards, '
      f'{ITERATIONS} iterations/combo, writing to {SHARED_FOLDER}')

In [ ]:
# Clone the code straight from GitHub — no zip, no manual upload. The repo
# is public, so this works with no authentication.
import os

PROJECT_DIR = '/content/martin'
REPO_URL = 'https://github.com/samdotbin/martin.git'

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
else:
    print(f'{PROJECT_DIR} already exists — pulling the latest instead of re-cloning.')
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}

In [ ]:
# data/raw's 182MB of historical CSVs isn't in git (see the repo's
# .gitignore) — it's attached to a GitHub Release instead. Downloads once;
# safe to re-run (skips if already present, e.g. after a disconnect where
# PROJECT_DIR survived but a fresh clone would still need this).
import urllib.request
import zipfile

DATA_URL = 'https://github.com/samdotbin/martin/releases/download/data-v1/forex_rl_v4_data_raw.zip'
data_dir = f'{PROJECT_DIR}/data/raw'

if os.path.isdir(data_dir) and len([f for f in os.listdir(data_dir) if f.endswith('.csv')]) >= 20:
    print(f'{data_dir} already has the data — skipping download.')
else:
    os.makedirs(data_dir, exist_ok=True)
    print('downloading historical price data (~43MB)...')
    tmp_zip = f'{data_dir}/_download.zip'
    urllib.request.urlretrieve(DATA_URL, tmp_zip)
    with zipfile.ZipFile(tmp_zip) as z:
        z.extractall(data_dir)
    os.remove(tmp_zip)
    n_csvs = len([f for f in os.listdir(data_dir) if f.endswith('.csv')])
    print(f'done — {n_csvs} CSV(s) in {data_dir}')

In [ ]:
# MetaTrader5 is Windows-only and gated by an environment marker in
# requirements.txt (`; platform_system == "Windows"`) — pip skips it
# automatically here, and nothing in the training path imports it.
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU visible. Runtime -> Change runtime type -> GPU, then re-run this cell."
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Sanity check first — same coverage report you'd run locally.
!python data_pipeline.py

## Optional: bump ROLLOUT_N_ENVS for GPU

`config.ROLLOUT_N_ENVS` (default 32) sets how many environments are batched
into one model forward call. The model is small (2-5M params) and Colab GPUs
have plenty of headroom, so a higher value amortizes kernel-launch overhead
further. Edit `config.py` directly (or uncomment below) if you want to try
64 or 128 — there's no single right answer, and changing it does change the
exact (though not the statistical) outcome for a given seed, so pick a value
and keep it fixed for the whole sweep.

In [ ]:
# import config
# print('current ROLLOUT_N_ENVS:', config.ROLLOUT_N_ENVS)
# Edit config.py's ROLLOUT_N_ENVS value directly instead of monkey-patching
# here, since train.py runs as a subprocess below and won't see an in-notebook
# variable change.

In [ ]:
# Claim your shard indices from the shared registry — the system figures
# out which ones are still free, you don't need to be handed a range
# manually. Safe to re-run: idempotent for the SAME MY_NAME (returns your
# existing claims first, only grabs new ones for any shortfall).
import sys
sys.path.insert(0, PROJECT_DIR)
from scripts.claim_shards import claim

MY_SHARDS = claim(SHARED_FOLDER, TOTAL_SHARDS, N_SHARDS_WANTED, MY_NAME)
print(f'{MY_NAME}: claimed shards {MY_SHARDS}')

In [ ]:
import subprocess
import time

# Each of YOUR processes gets its own subfolder under the SHARED Drive
# folder (via env var overrides — see config.py) so concurrent processes —
# yours or anyone else's, in case sessions overlap — never read-modify-write
# the SAME RUN_MANIFEST.json at once. All processes share the one local
# code+data checkout (PROJECT_DIR) — data/raw is read-only.
os.makedirs(SHARED_FOLDER, exist_ok=True)

procs = []
for idx in MY_SHARDS:
    shard_dir = f'{SHARED_FOLDER}/shard{idx}'
    ckpt_dir = f'{shard_dir}/checkpoints'
    runs_dir = f'{shard_dir}/runs'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)

    env = os.environ.copy()
    env['FOREX_RL_CHECKPOINT_DIR'] = ckpt_dir
    env['FOREX_RL_RUNS_DIR'] = runs_dir

    log_path = f'{shard_dir}/train_log.txt'
    p = subprocess.Popen(
        ['python', 'train.py', '--resume', '--iterations', str(ITERATIONS),
         '--shard-index', str(idx), '--shard-count', str(TOTAL_SHARDS)],
        cwd=PROJECT_DIR, env=env,
        stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
    )
    procs.append({'idx': idx, 'proc': p, 'log': log_path})
    print(f'started shard {idx} (pid {p.pid}) -> {log_path}')
    time.sleep(2)  # stagger starts slightly so they don't all hit disk/data-loading at once

print(f'\n{len(procs)} shard(s) running in the background. Use the next cell '
      f'(re-run it any time) to check progress.')

In [ ]:
# Push YOUR results straight to GitHub automatically, in the background,
# every few minutes — this is what makes the owner's dashboard pick you up
# with no publish step on anyone's part. Uses a token the project owner
# placed in the shared folder (github_token.txt); nothing to set up here.
# Also pushes a small heartbeat (contributions/{MY_NAME}/heartbeat.json —
# name, shards, timestamp) every cycle, which is what makes you show up on
# the Contribute tab's "Who's contributing" list — that list reads this
# straight out of the repo, so it needs nothing from the project owner
# either. Safe to re-run: starts one background thread, skips relaunching
# if already running. The token is read fresh EVERY cycle (not once at
# startup) so a missing-token-file error just prints and retries next
# cycle instead of silently killing the whole background thread — if the
# owner adds the token file after training has already started, the very
# next cycle picks it up with nothing to re-run.
import sys
import threading
import time

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from scripts.push_to_github import push_all, push_heartbeat, read_shared_token

GITHUB_REPO = 'samdotbin/martin'
PUSH_INTERVAL_SECONDS = 300  # 5 minutes

def _auto_push_loop():
    while True:
        try:
            token = read_shared_token(SHARED_FOLDER)
            pushed = push_all(GITHUB_REPO, token, MY_NAME, SHARED_FOLDER, MY_SHARDS)
            push_heartbeat(GITHUB_REPO, token, MY_NAME, MY_SHARDS)
            if pushed:
                print(f'[{time.strftime("%H:%M:%S")}] pushed {len(pushed)} file(s) to GitHub')
        except Exception as e:
            print(f'[{time.strftime("%H:%M:%S")}] auto-push failed (will retry in '
                  f'{PUSH_INTERVAL_SECONDS // 60} min): {e}')
        time.sleep(PUSH_INTERVAL_SECONDS)

if '_auto_push_thread' in dir() and _auto_push_thread.is_alive():
    print('auto-push already running — not starting a second thread.')
else:
    _auto_push_thread = threading.Thread(target=_auto_push_loop, daemon=True)
    _auto_push_thread.start()
    print(f'background auto-push started — pushing to GitHub every {PUSH_INTERVAL_SECONDS // 60} min. '
          f'Leave this tab open; closing it stops the thread (your last push still counts).')

In [ ]:
# Re-run this cell any time to check progress. All "exited (code 0)" means
# every shard finished — move on to the merge cell at the bottom.
for entry in procs:
    rc = entry['proc'].poll()
    status = 'running' if rc is None else f'exited (code {rc})'
    print(f"shard {entry['idx']} (pid {entry['proc'].pid}): {status}")
    !tail -n 3 "{entry['log']}"
    print()

# Resource check — see the parallelism config cell for what to watch for.
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv
!uptime

## Merge and archive

Every contributor's process wrote to their own `shard{i}/` subfolder under
the ONE shared Drive folder. This pulls in ALL `TOTAL_SHARDS` of them —
everyone's combined progress, not just yours — into one consolidated
`checkpoints/`+`runs/` in `PROJECT_DIR`, and zips that up for download.
Safe to run any time, by anyone with access to the shared folder, even
mid-training.

In [ ]:
import datetime

all_shard_dirs = ' '.join(f'{SHARED_FOLDER}/shard{idx}' for idx in range(TOTAL_SHARDS))
!python scripts/merge_shard_results.py {all_shard_dirs}

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = f'/content/drive/MyDrive/forex_rl_v4_merged_results_{stamp}.zip'
!zip -rq "{archive_path}" checkpoints runs
print('wrote', archive_path)